# RuralDoc GRPO Training
**Meta PyTorch x HuggingFace OpenEnv Hackathon**

Trains `meta-llama/Llama-3.1-8B-Instruct` using GRPO on the RuralDoc rural Indian PHC clinical diagnosis environment.

The agent learns to:
- Order the right diagnostic tests
- Diagnose diseases correctly (malaria, dengue, TB, typhoid etc.)
- Stay within budget
- Refer when necessary

**Runtime: T4 GPU (~90 mins) or A100 (~30 mins)**

## Step 1 — Install Dependencies

In [ ]:
%%capture
!pip install unsloth trl transformers datasets accelerate bitsandbytes peft openai -q
!pip install torch --upgrade -q

## Step 2 — Clone RuralDoc Repo

In [ ]:
!git clone https://github.com/DeshnaDey/RuralDoc-v1 /content/RuralDoc
%cd /content/RuralDoc
import sys
sys.path.insert(0, '/content/RuralDoc')
print('Repo cloned and path set.')

## Step 3 — Set Environment Variables

In [ ]:
import os

# ⚠️ SET YOUR HF TOKEN HERE
os.environ['HF_TOKEN'] = 'YOUR_HF_TOKEN_HERE'  # ⚠️ Replace with your token before running
os.environ['MODEL_NAME'] = 'meta-llama/Llama-3.1-8B-Instruct'
os.environ['MAX_STEPS'] = '200'
os.environ['BATCH_SIZE'] = '2'
os.environ['OUTPUT_DIR'] = '/content/ruraldoc-grpo'

from huggingface_hub import login
login(token=os.environ['HF_TOKEN'])
print('HF login successful.')

## Step 4 — Load Model with Unsloth

In [ ]:
from unsloth import FastLanguageModel
import torch

MAX_SEQ_LENGTH = 1024

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name='meta-llama/Llama-3.1-8B-Instruct',
    max_seq_length=MAX_SEQ_LENGTH,
    load_in_4bit=True,
    token=os.environ['HF_TOKEN'],
)

# Add LoRA adapters
model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj',
                    'gate_proj', 'up_proj', 'down_proj'],
    lora_alpha=16,
    lora_dropout=0,
    bias='none',
    use_gradient_checkpointing='unsloth',
    random_state=42,
)

print(f'Model loaded. Parameters: {model.num_parameters():,}')

## Step 5 — Build Dataset from RuralDoc Scenarios

In [ ]:
import json
import re
from datasets import Dataset
from env.scenarios import scenarios_v2

TRAIN_SCENARIOS = scenarios_v2[:20]

SYSTEM_PROMPT = """You are RuralDoc — an AI clinical decision-support agent for rural Indian PHCs.
Diagnose patients using only basic PHC tools on a limited budget.

Available action types:
1. Order a test: {\"type\": \"order_test\", \"test_name\": \"<name>\"}
2. Make diagnosis: {\"type\": \"diagnose\", \"diagnosis\": \"<disease_name>\"}
3. Refer patient: {\"type\": \"refer\", \"reason\": \"<reason>\"}

Always respond with a single valid JSON object. Be efficient — budget is limited."""

def scenario_to_prompt(sc):
    p = sc.get('patient', {})
    symptoms = '\n'.join(f'  • {s}' for s in sc.get('presenting_symptoms', []))
    vitals = sc.get('vitals', {})
    tests = '\n'.join(f'  • {n} [{c} units]' for n, c in sc.get('test_costs', {}).items())
    return f"""PATIENT
  Age: {p.get('age','?')}  Gender: {p.get('gender','?')}  Location: {p.get('location','?')}
SYMPTOMS
{symptoms}
VITALS
  Temp: {vitals.get('temperature','?')}  BP: {vitals.get('bp','?')}  HR: {vitals.get('hr','?')}
BUDGET: {sc.get('budget', 20)} units
AVAILABLE TESTS
{tests}

Respond with a single JSON action object."""

data = []
for _ in range(10):
    for sc in TRAIN_SCENARIOS:
        data.append({
            'prompt': [
                {'role': 'system', 'content': SYSTEM_PROMPT},
                {'role': 'user', 'content': scenario_to_prompt(sc)},
            ],
            'scenario_id': sc['id'],
        })

dataset = Dataset.from_list(data)
print(f'Dataset size: {len(dataset)} examples')

## Step 6 — Define Reward Function

In [ ]:
from env.rewards import calculate_reward

scenarios_map = {sc['id']: sc for sc in TRAIN_SCENARIOS}

def compute_reward(response, scenario):
    try:
        match = re.search(r'\{[^{}]*\}', response, re.DOTALL)
        if not match:
            return -0.15
        action = json.loads(match.group())
    except json.JSONDecodeError:
        return -0.15

    current_state = {
        'current_day': 1,
        'budget_remaining': scenario.get('budget', 20),
        'tests_ordered': [],
        'referred': False,
    }
    try:
        return float(calculate_reward(current_state, action, scenario))
    except Exception as e:
        print(f'reward error: {e}')
        return -0.05

def reward_fn(completions, prompts=None, **kwargs):
    rewards = []
    scenario_ids = kwargs.get('scenario_id', [])
    for i, completion in enumerate(completions):
        text = completion[0].get('content', '') if isinstance(completion, list) else str(completion)
        sc_id = scenario_ids[i] if i < len(scenario_ids) else TRAIN_SCENARIOS[0]['id']
        sc = scenarios_map.get(sc_id, TRAIN_SCENARIOS[0])
        rewards.append(compute_reward(text, sc))
    return rewards

# Quick sanity check
test_response = '{"type": "diagnose", "diagnosis": "malaria"}'
print(f'Sanity check reward: {compute_reward(test_response, TRAIN_SCENARIOS[0])}')

## Step 7 — Train with GRPO

In [ ]:
from trl import GRPOConfig, GRPOTrainer

config = GRPOConfig(
    output_dir='/content/ruraldoc-grpo',
    num_train_epochs=1,
    max_steps=200,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    learning_rate=5e-6,
    logging_steps=10,
    save_steps=50,
    warmup_steps=20,
    bf16=True,
    remove_unused_columns=False,
    report_to='none',
    max_completion_length=256,
    num_generations=4,
    temperature=0.8,
)

trainer = GRPOTrainer(
    model=model,
    args=config,
    train_dataset=dataset,
    reward_funcs=reward_fn,
    processing_class=tokenizer,
)

print('Starting GRPO training...')
trainer.train()
print('Training complete!')

## Step 8 — Plot Reward Curve

In [ ]:
import matplotlib.pyplot as plt
import json

with open('/content/ruraldoc-grpo/trainer_state.json') as f:
    state = json.load(f)

steps   = [x['step']   for x in state['log_history'] if 'reward' in x]
rewards = [x['reward'] for x in state['log_history'] if 'reward' in x]

plt.figure(figsize=(12, 5))
plt.plot(steps, rewards, color='#5aada8', linewidth=2.5, label='Mean Reward')
plt.axhline(y=0, color='gray', linestyle='--', alpha=0.5)
plt.fill_between(steps, rewards, 0, alpha=0.15, color='#5aada8')
plt.title('RuralDoc GRPO Training — Reward Curve', fontsize=14, fontweight='bold')
plt.xlabel('Training Steps')
plt.ylabel('Mean Reward')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('/content/reward_curve.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved reward_curve.png')

## Step 9 — Save & Push to HuggingFace Hub

In [ ]:
trainer.save_model('/content/ruraldoc-grpo')
tokenizer.save_pretrained('/content/ruraldoc-grpo')

# Push to HF Hub
model.push_to_hub('Kiddy007/ruraldoc-llama3-grpo', token=os.environ['HF_TOKEN'])
tokenizer.push_to_hub('Kiddy007/ruraldoc-llama3-grpo', token=os.environ['HF_TOKEN'])
print('Model pushed to HuggingFace Hub: Kiddy007/ruraldoc-llama3-grpo')